# NumPy support in Numba

**Optional deep dive, about 15 minutes.**

Numba complements NumPy by compiling array-aware loops and custom elementwise functions. This notebook explores two extensions to the core lesson:

- How dtype and memory layout create specialized compiled signatures.
- How `@vectorize` creates a custom compiled ufunc.

As in the core notebook, verify each result and warm up every new signature before timing.

In [ ]:
import numpy as np
import numba
from numba import njit

print("NumPy", np.__version__)
print("Numba", numba.__version__)

## Numba specialization by dtype

Numba generates a specialized implementation for each input signature. The signature includes argument dtypes, dimensions, and array layout. Consider a function that clamps values to zero below a threshold:

In [ ]:
@njit
def zero_clamp(x, threshold):
    # This function is designed for one-dimensional arrays.
    out = np.empty_like(x)
    for i in range(out.shape[0]):
        if np.abs(x[i]) > threshold:
            out[i] = x[i]
        else:
            out[i] = 0
    return out

In [ ]:
a_small = np.linspace(0, 1, 50)
zero_clamp(a_small, 0.3)

We will compare several array signatures:

- `int64` with contiguous storage.
- `float32` with contiguous storage.
- `float32` with a stride, so elements are not contiguous in memory.

In [ ]:
n = 10000
a_int64 = np.arange(n, dtype=np.int64)
a_float32 = np.linspace(0, 1, n, dtype=np.float32)
a_float32_strided = np.linspace(0, 1, 2 * n, dtype=np.float32)[::2]

cases = (
    (a_int64, 1600),
    (a_float32, 0.3),
    (a_float32_strided, 0.3),
)

# Compile, warm up, and verify every signature before timing it.
for values, threshold in cases:
    actual = zero_clamp(values, threshold)
    expected = np.where(np.abs(values) > threshold, values, 0)
    np.testing.assert_array_equal(actual, expected)

In [ ]:
%timeit -n 10 -r 3 zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 zero_clamp(a_float32_strided, 0.3)

The arrays contain the same number of elements, but dtype and memory layout can change performance. Inspect the compiled signatures instead of assuming that one machine-code version handles every input:

In [ ]:
zero_clamp.signatures

Numba array signatures have the form `array(dtype, dimensions, layout)`. The first signature comes from `a_small`, a contiguous one-dimensional `float64` array. The next signatures cover contiguous `int64`, contiguous `float32`, and strided `float32` inputs. A strided view commonly receives the flexible `A` layout because it is neither C-contiguous nor Fortran-contiguous.

Compare with a clear NumPy implementation. Numba may benefit from specialization and from avoiding temporary arrays, but the benchmark decides:

In [ ]:
def np_zero_clamp(x, threshold):
    return np.where(np.abs(x) > threshold, x, 0)

In [ ]:
%timeit -n 10 -r 3 np_zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 np_zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 np_zero_clamp(a_float32_strided, 0.3)

## Creating ufuncs

Universal functions, usually called ufuncs, broadcast an elementwise operation across arrays. NumPy provides many compiled ufuncs, and Numba can create custom ones with `@vectorize`.

In [ ]:
from numba import vectorize

In [ ]:
@vectorize
def ufunc_zero_clamp(x, threshold):
    if np.abs(x) > threshold:
        return x
    else:
        return 0

In [ ]:
for values, threshold in cases:
    actual = ufunc_zero_clamp(values, threshold)
    expected = np_zero_clamp(values, threshold)
    np.testing.assert_array_equal(actual, expected)

%timeit -n 10 -r 3 ufunc_zero_clamp(a_int64, 1600)
%timeit -n 10 -r 3 ufunc_zero_clamp(a_float32, 0.3)
%timeit -n 10 -r 3 ufunc_zero_clamp(a_float32_strided, 0.3)

For this simple operation, the custom ufunc may be no faster than the manual compiled loop or NumPy. NumPy already uses compiled ufuncs. Numba `@vectorize` is most useful when it expresses a custom elementwise operation that is not already a short combination of existing NumPy operations.

## Takeaway

- A new dtype or layout can trigger a new Numba compilation.
- Warm up and verify every signature before comparing steady-state timing.
- Prefer clear NumPy when it already provides the operation you need.